# BERT fine-tune: finansal belgelerde PII tespiti (COE025 Proje 2)

Bu notebook `train_bert_ner.py` script'ini çalıştırır; eğitim kodu script'te, burada tekrar yazılmaz.

**Hazırlık:** `train_bert_ner.py` ve bu notebook Drive'da aynı klasörde olsun: `MyDrive/NLP-2/`.
Runtime → Change runtime type → **GPU**.

Adımlar: 1) GPU kontrolü · 2) Drive · 3) repo + script · 4) TAM eğitim · 5) sonuçlara bakış.

## 1) GPU var mı?

In [1]:
!nvidia-smi

Fri Oct 23 16:53:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P0             45W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 2) Drive'ı bağla
Sonuçlar Drive'a yazılır, oturum kapansa da kaybolmaz.

In [4]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/NLP-2'   # script ve sonuçların klasörü
import os
assert os.path.exists(f'{DRIVE}/train_bert_ner.py'), 'train_bert_ner.py bu klasörde yok'

Mounted at /content/drive


## 3) Repoyu klonla (veri burada) ve script'i içine koy
Script repoya pushlanınca kopyalama satırı gereksiz olacak.

In [3]:
%cd /content
!rm -rf nlp-project-2
!git clone -q https://github.com/ata-sesli/nlp-project-2
%cd /content/nlp-project-2
!git log --oneline -1
!mkdir -p bert && cp "$DRIVE/train_bert_ner.py" bert/
!pip install -q --use-pep517 seqeval
!ls dataset/data

/content
/content/nlp-project-2
e933dff (HEAD -> main, origin/main, origin/HEAD) Add CRF model and Enes's contributions file
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
labels.json  REPORT.md	test.jsonl.gz  train.jsonl.gz  val.jsonl.gz


## 4) TAM eğitim
Tüm train (30.545 parça), 3 epoch, en iyi checkpoint val varlık F1'ine göre.
İlk epoch sonunda `eval_f1` satırına bak: 0,7 ve üstüyse her şey yolunda. Bitince 5. adımı çalıştır.

In [4]:
!python bert/train_bert_ner.py --out "$DRIVE/bert_full"

device=cuda model=bert-base-cased train=30545 val=3389 test=3903
config.json: 100% 570/570 [00:00<00:00, 3.28MB/s]
tokenizer_config.json: 100% 49.0/49.0 [00:00<00:00, 312kB/s]
vocab.txt: 100% 213k/213k [00:00<00:00, 9.67MB/s]
tokenizer.json: 100% 436k/436k [00:00<00:00, 34.2MB/s]
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (544 > 512). Running this sequence through the model will result in indexing errors
chunks longer than 512 subwords, split in pieces: train 1243, val 139
words cut by the 512-token limit: train 2, val 0  (should be 0)

model.safetensors: downloading bytes:  82% 356M/436M [00:01<00:00, 439MB/s, 28.5MB/s  ]
model.safetensors: downloading bytes:  94% 411M/436M [00:01<00:00, 372MB/s, 37.4MB/s  ]
model.safetensors: downloading bytes: 100% 413M/413M [00:01<00:00, 244MB/s, 38.3MB/s  ]
model.safetensors: reconstructing file: 100% 436M/436M [00:01<00:00, 258MB/s, 41.8MB/s  ]
Loading weights: 100% 197/197 [00

## 5) Sonuçlara bakış
Epoch başına val F1 (3 epoch yetti mi?), val/test varlık F1, kaçırılan PII oranı ve tür bazında F1.
Dosyalar: `bert_full/results.json`, `bert_full/predictions/val.jsonl`, `test.jsonl` (benchmark için), `bert_full/model/`.

In [5]:
import json, pandas as pd
RUN = f'{DRIVE}/bert_full'
r = json.load(open(f'{RUN}/results.json'))
print('GPU:', r['gpu'], '| train chunks:', r['train_chunks'], '| train time (s):', r['train_seconds'])
print('words cut by max_len (should be 0):', r['words_cut_by_max_len'], '| chunks split:', r['chunks_split_for_max_len'])
for h in r['val_history']:
    print(f"epoch {h['epoch']:.0f}: val entity F1 {h['f1']:.4f} | loss {h['loss']:.4f}")
for split in ('val', 'test'):
    s = r['splits'][split]
    print(f"{split}: entity F1 {s['f1']:.4f} | P {s['precision']:.4f} | R {s['recall']:.4f} | "
          f"missed PII {s['missed_pii_rate']:.1%} | token acc {s['token_accuracy']:.4f}")
types = {k: v for k, v in r['splits']['test']['per_type'].items() if k.isupper()}
pd.DataFrame(types).T[['precision', 'recall', 'f1-score', 'support']].sort_values('f1-score').round(4)

GPU: NVIDIA A100-SXM4-40GB | train chunks: 30545 | train time (s): 453.6
words cut by max_len (should be 0): {'train': 2, 'val': 0} | chunks split: {'train': 1243, 'val': 139}
epoch 1: val entity F1 0.8389 | loss 0.0487
epoch 2: val entity F1 0.8575 | loss 0.0428
epoch 3: val entity F1 0.8613 | loss 0.0418
val: entity F1 0.8612 | P 0.8535 | R 0.8690 | missed PII 13.1% | token acc 0.9866
test: entity F1 0.8683 | P 0.8595 | R 0.8772 | missed PII 12.3% | token acc 0.9872


,precision,recall,f1-score,support
COMPANY,0.8053,0.8462,0.8252,2932.0
DATE,0.8473,0.8925,0.8693,5712.0
ACCOUNT,0.9113,0.8403,0.8744,526.0
NAME,0.9040,0.8520,0.8772,3447.0
ADDRESS,0.8753,0.9041,0.8895,1856.0
PHONE,0.8785,0.9228,0.9001,337.0
ID,0.9016,0.9347,0.9178,490.0
EMAIL,0.9116,0.9419,0.9265,241.0


## 6) Hugging Face'e yükleme
Model `bert_full/model/` klasöründen Hub'a yüklenir; model kartı `results.json`'dan yazılır.
Giriş için **Write** yetkili bir token gerekir. Token sadece açılan kutuya yapıştırılır, hiçbir hücreye yazılmaz.

In [1]:
from huggingface_hub import notebook_login
notebook_login()

In [6]:
REPO_ID = 'akifkulll/finance-pii-bert-base-cased'
!python "$DRIVE/push_to_hub.py" --model-dir "$DRIVE/bert_full/model" --results "$DRIVE/bert_full/results.json" --repo-id "$REPO_ID"

---
license: apache-2.0
language: en
base_model: bert-base-cased
datasets:
- gretelai/synthetic_pii_finance_multilingual
pipeline_tag: token-classification
tags:
- ner
- pii
- token-classification
---

# finance-pii-bert-base-cased

`bert-base-cased` fine-tuned to find personal data (PII) in financial documents, as token
classification with BIO tags. 8 entity types (17 tags): NAME, COMPANY, ADDRESS, DATE, EMAIL,
PHONE, ACCOUNT, ID.

Built for the COE025 Natural Language Processing course project (İstinye University).
Code and data preparation: https://github.com/ata-sesli/nlp-project-2

## Training

- Data: English part of `gretelai/synthetic_pii_finance_multilingual`, cleaned and converted to
  word-level BIO by the project's `dataset/prepare_dataset.py`; 30,545 training chunks
  of up to 200 words.
- 3 epochs, learning rate 3e-05, 10% linear warm-up, batch size
  16, fp16, seed 42; best epoch chosen by validation entity F1.
- Labels go on the first subword of each word; chunks longer